# Comparable-sales feature ablation + importance — Sep–Nov 2025

**Goal:** Test whether all 14 comparable statistics improve off-market XGBoost predictions relative to smaller subsets, while preserving the **December 2025–May 2026 final validation**.

- Fixed accepted reference XGBoost configuration, including its center-target treatment, estimator parameters, text/spline/geographic features.
- Walk-forward: train on sales **strictly before** the month; predict September, October, November 2025 separately.
- Train identical estimators across five comparable-column subsets. No modification to `Dean/`, `run.py`, or your winning ensembles.
- Built-in XGBoost *total gain* feature ranking (all fitted columns) and **held-out permutation importance** on comparable features (only full model).
- Month-level and pooled MdAPE/MAPE; paired, month-stratified bootstrap CIs. Save predictions and failures for inspection.

**Interpretation:** Permutation importance is affected by correlated features. A retrained ablation is more decisive for feature removal. This is a model-selection window, not an untouched final evaluation.

In [1]:
from __future__ import annotations
import copy
import json
import sys
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from xgboost import XGBRegressor

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data.py").exists() and (p / "ensemble_method/models/models.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Launch Jupyter within the repository containing data.py and ensemble_method/models/models.py")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT))  # supports legacy 'features' imports inside ensemble_method.models
import data
from ensemble_method.models.models import make_model
from ensemble_method.features.pipeline import FeaturePipeline

MONTHS = pd.period_range("2025-09", "2025-11", freq="M")
OUTPUT = ROOT / "outputs/experiments/comparable_feature_ablation_sep_nov_2025"
OUTPUT.mkdir(parents=True, exist_ok=True)
SEED = 42
BOOTSTRAP_REPEATS = 1500
PERMUTATION_REPEATS = 3
RUN_PERMUTATION = True  # Set False to skip extra predictions
CACHE_PREDICTIONS = False  # Set True only if data/configuration have not changed between runs
print("Project:", ROOT)
print("Output:", OUTPUT)

Project: c:\Users\deant\Desktop\Coding\Data Dash final
Output: c:\Users\deant\Desktop\Coding\Data Dash final\outputs\experiments\comparable_feature_ablation_sep_nov_2025


## 1. Freeze the accepted reference model and feature subsets

The accepted XGBoost reference is obtained from the saved off-market roster. No substitute hyperparameters or new tuning are introduced. All subsets use **exactly the same computed feature matrix**, with only comparable columns removed afterwards.

In [2]:
reference = make_model("off_market").models["selected_47_splines_geo_reference"]
assert len(reference.specs) == 1, "Expected a one-component winning reference model"
spec = reference.specs[0]
assert spec.estimator == "xgboost"
assert spec.row_mode == "all", "This experiment expects all training rows"
print("Reference features:", spec.features)
print("Target median-centered:", reference.center_target)
print("Estimator settings:", spec.params)

COMPS = [
    "comp_log_price", "comp_log_ppsf", "comp_local_ppsf_median",
    "comp_price_std", "comp_ppsf_std", "comp_count",
    "comp_effective_n", "comp_nearest_km", "comp_mean_distance_km",
    "comp_mean_age_days", "comp_same_type_share", "comp_area_similarity",
    "comp_recent_180d_share", "comp_momentum_recent_older",
]
CORE = ["comp_log_price", "comp_log_ppsf", "comp_count",
        "comp_effective_n", "comp_same_type_share"]
CORE_PLUS = CORE + ["comp_price_std", "comp_ppsf_std", "comp_nearest_km",
                    "comp_mean_distance_km", "comp_area_similarity"]
KEEP = {
    "A_full_14": COMPS,
    "B_none_0": [],
    "C_core_5": CORE,
    "D_core_plus_10": CORE_PLUS,
    "E_no_momentum_13": [x for x in COMPS if x != "comp_momentum_recent_older"],
}
for key, values in KEEP.items():
    assert len(values) == int(key.split("_")[-1]), (key, values)
    print(f"{key:23s}: {len(values):2d} comparable columns")

# Feature fit/train/test must use identical historical cutoffs throughout.
listings = data.load("listings").copy()
listings["date_sold"] = pd.to_datetime(listings["date_sold"], errors="raise")
listings["date_listed"] = pd.to_datetime(listings["date_listed"], errors="raise")
assert listings["ml_num"].is_unique, "Expected unique property IDs"
folds = {}
for month in MONTHS:
    train = listings.loc[listings.date_sold < month.start_time].copy()
    holdout = listings.loc[listings.date_sold.dt.to_period("M") == month].copy()
    assert len(train) and len(holdout)
    assert train.date_sold.max() < month.start_time
    folds[str(month)] = (train, holdout)
    print(f"{month}: training {len(train):,}, held out {len(holdout):,}")

Reference features: {'text_mode': 'selected', 'splines': True, 'geo_context': True, 'comparable': 'base', 'max_extra_terms': 1000, 'min_df': 10}
Target median-centered: True
Estimator settings: {'n_estimators': 700, 'learning_rate': 0.09522271657253033, 'max_depth': 8, 'min_child_weight': 16.684619386293466, 'reg_lambda': 19.50331236061207, 'reg_alpha': 0.0, 'subsample': 0.9804685587557792, 'colsample_bytree': 0.8994749947027713, 'tree_method': 'hist', 'grow_policy': 'depthwise', 'device': 'cpu', 'enable_categorical': True, 'objective': 'reg:squarederror', 'random_state': 42, 'n_jobs': -1, 'verbosity': 0}
A_full_14              : 14 comparable columns
B_none_0               :  0 comparable columns
C_core_5               :  5 comparable columns
D_core_plus_10         : 10 comparable columns
E_no_momentum_13       : 13 comparable columns


2025-09: training 133,779, held out 2,035
2025-10: training 135,814, held out 2,250
2025-11: training 138,064, held out 1,867


## 2. Fit the five configurations per month

Training loss is computed on the *training matrix*, while holdout predictions use `data.hide_answers` and the **same fitted feature pipeline**. A failure in one variant doesn't prevent the others from running. Results are saved progressively.

In [3]:
def score(y_true, predicted):
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(predicted, dtype=float)
    assert p.shape == y.shape and np.isfinite(p).all() and (p > 0).all()
    ape = 100 * np.abs(p - y) / y
    return {"homes": len(y), "mape": float(ape.mean()),
            "mdape": float(np.median(ape)), "p90_ape": float(np.percentile(ape, 90)),
            "median_signed_error": float(np.median(100 * (p - y) / y))}

def new_estimator():
    params = copy.deepcopy(spec.params)
    params["random_state"] = spec.seed
    params.setdefault("enable_categorical", True)
    params.setdefault("tree_method", "hist")
    return XGBRegressor(**params)

def selected_matrix(X, kept):
    remove = [name for name in COMPS if name not in kept]
    return X.drop(columns=remove)

pred_rows, gain_rows, perm_rows, fit_rows, failures = [], [], [], [], []
for month, (train, holdout) in folds.items():
    print(f"\n========== {month} ==========", flush=True)
    try:
        pipeline = FeaturePipeline(model="xgboost", track="off_market", **spec.features)
        X_train = pipeline.fit_transform(train)
        hidden = data.hide_answers(holdout, "off_market")
        assert not {"sold_price", "date_sold", "list_price"}.intersection(hidden.columns)
        X_valid = pipeline.transform(hidden)
        assert X_train.columns.equals(X_valid.columns)
        assert set(COMPS).issubset(X_train.columns), "Some expected comparables are missing"
        y_train = np.log(train.sold_price.to_numpy(dtype=float))
        target_offset = float(np.median(y_train)) if reference.center_target else 0.0
        residual = y_train - target_offset
        y_valid = holdout.sold_price.to_numpy(dtype=float)
    except Exception as exc:
        failures.append({"month": month, "experiment": "FEATURE_PIPELINE", "error": repr(exc),
                         "traceback": traceback.format_exc()})
        print("FAILED pipeline:", repr(exc))
        continue

    for label, kept in KEEP.items():
        try:
            Xt = selected_matrix(X_train, kept)
            Xv = selected_matrix(X_valid, kept)
            assert Xt.columns.equals(Xv.columns)
            model = new_estimator()
            model.fit(Xt, residual)
            valid_pred = np.exp(target_offset + model.predict(Xv))
            train_pred = np.exp(target_offset + model.predict(Xt))
            valid_score = score(y_valid, valid_pred)
            train_score = score(train.sold_price.to_numpy(float), train_pred)
            print(f"{label:22s} train MdAPE={train_score['mdape']:.4f}%  "
                  f"holdout MdAPE={valid_score['mdape']:.4f}%", flush=True)

            pred_rows.append(pd.DataFrame({"month": month, "ml_num": holdout.ml_num.to_numpy(),
                                           "sold_price": y_valid, "prediction": valid_pred,
                                           "experiment": label}))
            fit_rows.append({"month": month, "experiment": label,
                             **{f"train_{k}": v for k,v in train_score.items()},
                             **{f"valid_{k}": v for k,v in valid_score.items()}})

            # Gain measures how much split-loss reduction a feature accrued, not a causal effect.
            gain = model.get_booster().get_score(importance_type="total_gain")
            gain_rows.extend({"month": month, "experiment": label, "feature": feature,
                              "total_gain": float(gain.get(feature, 0.0))}
                             for feature in Xt.columns)

            if RUN_PERMUTATION and label == "A_full_14":
                baseline = valid_score["mape"]
                for j, feature in enumerate(COMPS):
                    deltas = []
                    for rep in range(PERMUTATION_REPEATS):
                        rng = np.random.default_rng(SEED + 1000 * j + 100 * rep + MONTHS.get_loc(pd.Period(month)))
                        shuffled = Xv.copy()
                        shuffled[feature] = rng.permutation(Xv[feature].to_numpy())
                        changed_pred = np.exp(target_offset + model.predict(shuffled))
                        deltas.append(score(y_valid, changed_pred)["mape"] - baseline)
                    perm_rows.append({"month": month, "feature": feature,
                                      "delta_mape_pp": float(np.mean(deltas)),
                                      "repeat_sd_pp": float(np.std(deltas, ddof=1)) if len(deltas) > 1 else np.nan})
                print("  Permutation importance completed", flush=True)

            # Save incrementally in case a later experiment fails.
            pd.concat(pred_rows, ignore_index=True).to_csv(OUTPUT / "predictions.csv", index=False)
            pd.DataFrame(fit_rows).to_csv(OUTPUT / "fold_train_validation_scores.csv", index=False)
            pd.DataFrame(gain_rows).to_csv(OUTPUT / "total_gain_by_fold.csv", index=False)
            if perm_rows:
                pd.DataFrame(perm_rows).to_csv(OUTPUT / "permutation_by_fold.csv", index=False)
        except Exception as exc:
            failures.append({"month": month, "experiment": label, "error": repr(exc),
                             "traceback": traceback.format_exc()})
            print(f"FAILED {month}/{label}: {type(exc).__name__}: {exc}", flush=True)

(OUTPUT / "failures.json").write_text(json.dumps(failures, indent=2), encoding="utf-8")
print(f"\nCompleted: {len(fit_rows)}/15 variant-month fits; failures: {len(failures)}")
for f in failures:
    print(f"  {f['month']} / {f['experiment']}: {f['error']}")
if not pred_rows:
    raise RuntimeError("No valid model predictions. Inspect failures.json")
predictions = pd.concat(pred_rows, ignore_index=True)
fit_scores = pd.DataFrame(fit_rows)
gain_table = pd.DataFrame(gain_rows)
perm_table = pd.DataFrame(perm_rows)


========== 2025-09 ==========
A_full_14              train MdAPE=4.6249%  holdout MdAPE=7.5866%
  Permutation importance completed
B_none_0               train MdAPE=4.8132%  holdout MdAPE=7.5901%
C_core_5               train MdAPE=4.7680%  holdout MdAPE=7.6240%
D_core_plus_10         train MdAPE=4.6608%  holdout MdAPE=7.4771%
E_no_momentum_13       train MdAPE=4.6173%  holdout MdAPE=7.6902%

========== 2025-10 ==========
A_full_14              train MdAPE=4.6043%  holdout MdAPE=7.5067%
  Permutation importance completed
B_none_0               train MdAPE=4.7983%  holdout MdAPE=7.3240%
C_core_5               train MdAPE=4.7606%  holdout MdAPE=7.3655%
D_core_plus_10         train MdAPE=4.6421%  holdout MdAPE=7.2055%
E_no_momentum_13       train MdAPE=4.6251%  holdout MdAPE=7.1573%

========== 2025-11 ==========
A_full_14              train MdAPE=4.6187%  holdout MdAPE=7.9238%
  Permutation importance completed
B_none_0               train MdAPE=4.8230%  holdout MdAPE=7.5797%
C_core_5  

## 3. Pooled/monthly accuracy and paired comparisons

The paired bootstrap resamples **properties within each validation month** and then combines the sampled months, preserving the original number of homes per month. Positive Δ means worse than the full 14-feature reference.

In [4]:
def group_scores(rows):
    return pd.Series(score(rows.sold_price.to_numpy(float), rows.prediction.to_numpy(float)))

pooled = predictions.groupby("experiment", sort=False).apply(group_scores, include_groups=False).reset_index()
monthly = predictions.groupby(["month", "experiment"], sort=False).apply(group_scores, include_groups=False).reset_index()
pooled = pooled.sort_values("mdape").reset_index(drop=True)
print("POOLED: lower is better")
display(pooled.style.format({x: "{:.4f}%" for x in ["mape", "mdape", "p90_ape", "median_signed_error"]}))
print("BY MONTH")
display(monthly.sort_values(["month", "mdape"]))

# Exact pair matching prevents inadvertently comparing different houses.
wide = predictions.pivot(index=["month", "ml_num", "sold_price"],
                         columns="experiment", values="prediction").reset_index()
complete = wide.dropna(subset=list(KEEP)).copy()
print(f"Complete paired homes: {len(complete):,} / {sum(len(x[1]) for x in folds.values()):,}")

rng = np.random.default_rng(SEED)
def bootstrap_delta(new_prediction, base_prediction, actual, month_values, repeats=BOOTSTRAP_REPEATS):
    a = 100 * abs(new_prediction - actual) / actual
    b = 100 * abs(base_prediction - actual) / actual
    delta_mape = float(a.mean() - b.mean())
    delta_mdape = float(np.median(a) - np.median(b))
    partitions = [np.flatnonzero(month_values == m) for m in np.unique(month_values)]
    bootstrap = np.zeros((repeats, 2))
    for iteration in range(repeats):
        indices = np.concatenate([rng.choice(ix, size=len(ix), replace=True) for ix in partitions])
        bootstrap[iteration] = [a[indices].mean() - b[indices].mean(),
                                np.median(a[indices]) - np.median(b[indices])]
    ci = np.percentile(bootstrap, [2.5, 97.5], axis=0)
    return {"delta_mape_pp": delta_mape, "mape_ci_low": float(ci[0, 0]),
            "mape_ci_high": float(ci[1, 0]), "delta_mdape_pp": delta_mdape,
            "mdape_ci_low": float(ci[0, 1]), "mdape_ci_high": float(ci[1, 1])}

comparisons = []
for label in KEEP:
    if label == "A_full_14":
        continue
    comparisons.append({"experiment": label, "paired_homes": len(complete),
        **bootstrap_delta(complete[label].to_numpy(float), complete["A_full_14"].to_numpy(float),
                          complete.sold_price.to_numpy(float), complete.month.to_numpy())})
comparison_table = pd.DataFrame(comparisons).sort_values("delta_mdape_pp")
print("PAIRED DELTAS vs A_full_14 (negative is improvement)")
display(comparison_table)
pooled.to_csv(OUTPUT / "pooled_scores.csv", index=False)
monthly.to_csv(OUTPUT / "monthly_scores.csv", index=False)
comparison_table.to_csv(OUTPUT / "paired_bootstrap.csv", index=False)

POOLED: lower is better


,experiment,homes,mape,mdape,p90_ape,median_signed_error
0,B_none_0,6152.000000,9.8767%,7.4933%,21.0474%,1.6804%
1,D_core_plus_10,6152.000000,9.9885%,7.4971%,21.2068%,1.3461%
2,E_no_momentum_13,6152.000000,9.9758%,7.5386%,21.1467%,1.4584%
3,C_core_5,6152.000000,9.9437%,7.6346%,21.2875%,1.4944%
4,A_full_14,6152.000000,10.0062%,7.6458%,21.0736%,1.4799%


BY MONTH


,month,experiment,homes,mape,mdape,p90_ape,median_signed_error
3,2025-09,D_core_plus_10,2035.0,9.947283,7.477086,21.339969,0.593076
0,2025-09,A_full_14,2035.0,9.938776,7.586609,21.266342,0.571623
1,2025-09,B_none_0,2035.0,9.885724,7.590050,21.374869,1.420592
2,2025-09,C_core_5,2035.0,9.831715,7.624012,21.294739,0.996321
4,2025-09,E_no_momentum_13,2035.0,9.976531,7.690241,21.327248,0.877921
9,2025-10,E_no_momentum_13,2250.0,9.590850,7.157304,20.515434,0.526822
8,2025-10,D_core_plus_10,2250.0,9.614783,7.205534,20.281178,0.482682
6,2025-10,B_none_0,2250.0,9.578265,7.324047,19.750994,0.813852
7,2025-10,C_core_5,2250.0,9.672091,7.365521,20.648659,0.654718
5,2025-10,A_full_14,2250.0,9.686492,7.506729,19.995959,0.715896


Complete paired homes: 6,152 / 6,152
PAIRED DELTAS vs A_full_14 (negative is improvement)


,experiment,paired_homes,delta_mape_pp,mape_ci_low,mape_ci_high,delta_mdape_pp,mdape_ci_low,mdape_ci_high
0,B_none_0,6152,-0.129534,-0.247838,-0.012024,-0.152589,-0.375729,0.046610
2,D_core_plus_10,6152,-0.017661,-0.111894,0.072194,-0.148763,-0.322697,0.063097
3,E_no_momentum_13,6152,-0.030433,-0.125162,0.069610,-0.107201,-0.317469,0.049974
1,C_core_5,6152,-0.062471,-0.168117,0.038909,-0.011240,-0.190714,0.152460


## 4. Rank feature importance

**Total gain:** summed XGBoost split-gain for each input in each fold. Aggregate by normalizing within a fold, so fits with different total gains remain comparable. Ranking includes all inputs, and the separate comparable-only view highlights the 14 statistics.

**Permutation importance:** increase in held-out **MAPE percentage points** after shuffling one comparable column. Reported only for `A_full_14`. A negative/zero value does not prove the feature is useless, because correlated features can substitute for one another.

In [5]:
ranked_gain = []
for (month, experiment), g in gain_table.groupby(["month", "experiment"]):
    total = g.total_gain.sum()
    local = g.copy()
    local["gain_share"] = local.total_gain / total if total else 0.0
    ranked_gain.append(local)
normalized_gain = pd.concat(ranked_gain, ignore_index=True)
all_gain = (normalized_gain.groupby(["experiment", "feature"], as_index=False)
            .agg(mean_gain_share=("gain_share", "mean"), mean_total_gain=("total_gain", "mean")))
full_rank = (all_gain.loc[all_gain.experiment == "A_full_14"]
             .sort_values("mean_gain_share", ascending=False).reset_index(drop=True))
comp_rank = full_rank[full_rank.feature.isin(COMPS)].copy()
print("Top 25 features by mean total-gain share (full reference):")
display(full_rank.head(25))
print("14 comparable features by total-gain share:")
display(comp_rank)
all_gain.to_csv(OUTPUT / "gain_importance_all_variants.csv", index=False)
full_rank.to_csv(OUTPUT / "gain_importance_full_ranked.csv", index=False)
comp_rank.to_csv(OUTPUT / "gain_importance_comparables_ranked.csv", index=False)

if not perm_table.empty:
    perm_rank = (perm_table.groupby("feature", as_index=False)
                 .agg(mean_delta_mape_pp=("delta_mape_pp", "mean"),
                      min_month_delta_pp=("delta_mape_pp", "min"),
                      max_month_delta_pp=("delta_mape_pp", "max"))
                 .sort_values("mean_delta_mape_pp", ascending=False))
    print("Comparable permutation importance; positive = hurts MAPE when shuffled:")
    display(perm_rank)
    perm_rank.to_csv(OUTPUT / "permutation_importance_ranked.csv", index=False)
else:
    print("No permutation results (disabled or full-model fits failed).")

print("Saved results to:", OUTPUT)
print("Reminder: choose features on Sep–Nov only. Preserve Dec–May as final validation.")

Top 25 features by mean total-gain share (full reference):


,experiment,feature,mean_gain_share,mean_total_gain
0,A_full_14,comp_log_price,0.713887,139326.531250
1,A_full_14,community,0.104359,20367.180990
2,A_full_14,washrooms,0.033969,6628.143555
3,A_full_14,lot_front,0.030323,5920.036296
4,A_full_14,square_feet,0.015454,3013.077881
5,A_full_14,house_type_name,0.012055,2352.193766
6,A_full_14,lot_area,0.010921,2131.964111
7,A_full_14,comp_ppsf_std,0.004774,931.852417
8,A_full_14,parking,0.003781,738.018188
9,A_full_14,lot_depth,0.003301,643.118958


14 comparable features by total-gain share:


,experiment,feature,mean_gain_share,mean_total_gain
0,A_full_14,comp_log_price,0.713887,139326.531250
7,A_full_14,comp_ppsf_std,0.004774,931.852417
10,A_full_14,comp_local_ppsf_median,0.002673,521.877696
16,A_full_14,comp_price_std,0.001705,332.783539
17,A_full_14,comp_same_type_share,0.001447,282.564657
18,A_full_14,comp_area_similarity,0.001263,246.229009
19,A_full_14,comp_log_ppsf,0.001201,234.535090
21,A_full_14,comp_momentum_recent_older,0.001117,217.887334
27,A_full_14,comp_mean_distance_km,0.000948,184.997660
29,A_full_14,comp_effective_n,0.000805,157.240646


Comparable permutation importance; positive = hurts MAPE when shuffled:


,feature,mean_delta_mape_pp,min_month_delta_pp,max_month_delta_pp
5,comp_log_price,19.819467,19.147404,20.819015
10,comp_ppsf_std,0.354933,0.315609,0.385076
4,comp_log_ppsf,0.218496,0.155987,0.282903
9,comp_nearest_km,0.110018,0.076176,0.149029
3,comp_local_ppsf_median,0.101928,0.070950,0.138254
8,comp_momentum_recent_older,0.073543,0.060192,0.087704
13,comp_same_type_share,0.066964,0.030810,0.118131
11,comp_price_std,0.052408,0.022879,0.069468
7,comp_mean_distance_km,0.041245,0.033037,0.056887
12,comp_recent_180d_share,0.019168,0.013683,0.029707


Saved results to: c:\Users\deant\Desktop\Coding\Data Dash final\outputs\experiments\comparable_feature_ablation_sep_nov_2025
Reminder: choose features on Sep–Nov only. Preserve Dec–May as final validation.


In [6]:
# This cell depends only on the setup and reference model cells above (cells 1 and 3).
# It does NOT depend on running the original five-way fit cell.

FOLLOWUP_KEEP = {
    "F_price_only_1": ["comp_log_price"],
    "G_without_price_13": [col for col in COMPS if col != "comp_log_price"],
    "H_compact_no_price_4": [
        "comp_log_ppsf", "comp_ppsf_std", "comp_nearest_km", "comp_local_ppsf_median"
    ],
    "I_price_and_dispersion_2": ["comp_log_price", "comp_ppsf_std"],
}
assert {name.rsplit("_", 1)[-1] for name in FOLLOWUP_KEEP} == {"1", "13", "4", "2"}
assert all(len(cols) == len(set(cols)) and set(cols).issubset(COMPS)
           for cols in FOLLOWUP_KEEP.values())

FOLLOWUP_DIR = OUTPUT / "price_feature_followup"
FOLLOWUP_DIR.mkdir(parents=True, exist_ok=True)
FOLLOWUP_PRED_PATH = FOLLOWUP_DIR / "predictions_F_to_I.csv"
FOLLOWUP_FIT_PATH = FOLLOWUP_DIR / "fold_scores_F_to_I.csv"
FOLLOWUP_GAIN_PATH = FOLLOWUP_DIR / "gain_F_to_I.csv"
FOLLOWUP_FAILURE_PATH = FOLLOWUP_DIR / "failures_F_to_I.json"
FOLLOWUP_CACHE = True  # skip finished variant/month fits, reusing only matching saved predictions

# Keep these definitions local so the initial training cell does not have to be re-run.
def followup_score(y_true, predicted):
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(predicted, dtype=float)
    if y.shape != p.shape or not np.isfinite(p).all() or not (p > 0).all() or not (y > 0).all():
        raise ValueError("Expected aligned finite, strictly positive prices")
    ape = 100.0 * np.abs(p - y) / y
    return dict(homes=len(y), mape=float(np.mean(ape)), mdape=float(np.median(ape)),
                p90_ape=float(np.percentile(ape, 90)),
                median_signed_error=float(np.median(100.0 * (p - y) / y)))

def followup_estimator():
    params = copy.deepcopy(spec.params)
    params["random_state"] = spec.seed
    params.setdefault("enable_categorical", True)
    params.setdefault("tree_method", "hist")
    return XGBRegressor(**params)

def followup_matrix(X, kept):
    return X.drop(columns=[col for col in COMPS if col not in kept])

def read_cache(path):
    return pd.read_csv(path) if path.exists() and path.stat().st_size else pd.DataFrame()

print("New variants:")
for name, columns in FOLLOWUP_KEEP.items():
    print(f"  {name:30s}: {len(columns):2d} comparable columns")
print("Cache folder:", FOLLOWUP_DIR)

New variants:
  F_price_only_1                :  1 comparable columns
  G_without_price_13            : 13 comparable columns
  H_compact_no_price_4          :  4 comparable columns
  I_price_and_dispersion_2      :  2 comparable columns
Cache folder: c:\Users\deant\Desktop\Coding\Data Dash final\outputs\experiments\comparable_feature_ablation_sep_nov_2025\price_feature_followup


In [7]:
# Train only F/G/H/I on Sep/Oct/Nov 2025. Save every successful fit immediately.
# Existing A/B/C/D/E predictions and reports are NEVER overwritten.

previous = read_cache(FOLLOWUP_PRED_PATH) if FOLLOWUP_CACHE else pd.DataFrame()
previous_fits = read_cache(FOLLOWUP_FIT_PATH) if FOLLOWUP_CACHE else pd.DataFrame()
previous_gain = read_cache(FOLLOWUP_GAIN_PATH) if FOLLOWUP_CACHE else pd.DataFrame()

completed = set()
if not previous.empty and not previous_fits.empty:
    prediction_keys = set(zip(previous.month.astype(str), previous.experiment))
    fit_keys = set(zip(previous_fits.month.astype(str), previous_fits.experiment))
    completed = prediction_keys & fit_keys

new_predictions, new_fits, new_gains, followup_failures = [], [], [], []
for month, (train, holdout) in folds.items():
    needed = [label for label in FOLLOWUP_KEEP if (month, label) not in completed]
    if not needed:
        print(f"{month}: all 4 fits already cached")
        continue
    print(f"\n========== {month} (new variants only) ==========", flush=True)
    try:
        pipeline = FeaturePipeline(model="xgboost", track="off_market", **spec.features)
        X_train = pipeline.fit_transform(train)
        homes = data.hide_answers(holdout, "off_market")
        assert not {"sold_price", "date_sold", "list_price"}.intersection(homes.columns)
        X_valid = pipeline.transform(homes)
        assert X_train.columns.equals(X_valid.columns)
        assert set(COMPS).issubset(X_train.columns)
        y_log = np.log(train.sold_price.to_numpy(dtype=float))
        offset = float(np.median(y_log)) if reference.center_target else 0.0
        residual = y_log - offset
        train_actual = train.sold_price.to_numpy(dtype=float)
        valid_actual = holdout.sold_price.to_numpy(dtype=float)
    except Exception as exc:
        followup_failures.append(dict(month=month, experiment="PIPELINE", error=repr(exc),
                                      traceback=traceback.format_exc()))
        print(f"FAILED PIPELINE {month}: {exc}")
        continue

    for label in needed:
        try:
            Xt = followup_matrix(X_train, FOLLOWUP_KEEP[label])
            Xv = followup_matrix(X_valid, FOLLOWUP_KEEP[label])
            assert Xt.columns.equals(Xv.columns)
            model = followup_estimator()
            model.fit(Xt, residual)
            valid_pred = np.exp(offset + model.predict(Xv))
            train_pred = np.exp(offset + model.predict(Xt))
            train_metrics = followup_score(train_actual, train_pred)
            valid_metrics = followup_score(valid_actual, valid_pred)
            print(f"{label:30s} train MdAPE={train_metrics['mdape']:.4f}%  "
                  f"holdout MdAPE={valid_metrics['mdape']:.4f}%", flush=True)
            new_predictions.append(pd.DataFrame({
                "month": month, "ml_num": holdout.ml_num.to_numpy(),
                "sold_price": valid_actual, "prediction": valid_pred, "experiment": label,
            }))
            new_fits.append(dict(month=month, experiment=label,
                                 **{f"train_{k}": v for k,v in train_metrics.items()},
                                 **{f"valid_{k}": v for k,v in valid_metrics.items()}))
            gains = model.get_booster().get_score(importance_type="total_gain")
            new_gains.extend(dict(month=month, experiment=label, feature=col,
                                  total_gain=float(gains.get(col, 0.0))) for col in Xt.columns)

            # Persist after every completed variant; rerunning only needs unfinished fits.
            all_pred = pd.concat([previous, *new_predictions], ignore_index=True)
            all_fits = pd.concat([previous_fits, pd.DataFrame(new_fits)], ignore_index=True)
            all_gain = pd.concat([previous_gain, pd.DataFrame(new_gains)], ignore_index=True)
            all_pred.to_csv(FOLLOWUP_PRED_PATH, index=False)
            all_fits.to_csv(FOLLOWUP_FIT_PATH, index=False)
            all_gain.to_csv(FOLLOWUP_GAIN_PATH, index=False)
        except Exception as exc:
            followup_failures.append(dict(month=month, experiment=label,
                                          error=repr(exc), traceback=traceback.format_exc()))
            print(f"FAILED {month}/{label}: {type(exc).__name__}: {exc}", flush=True)

FOLLOWUP_FAILURE_PATH.write_text(json.dumps(followup_failures, indent=2), encoding="utf-8")
followup_predictions = read_cache(FOLLOWUP_PRED_PATH)
followup_fit_scores = read_cache(FOLLOWUP_FIT_PATH)
followup_gain = read_cache(FOLLOWUP_GAIN_PATH)
print(f"\nSaved {len(followup_fit_scores)}/12 follow-up fits; "
      f"new failures: {len(followup_failures)}")
for item in followup_failures:
    print(f"  {item['month']} / {item['experiment']}: {item['error']}")


========== 2025-09 (new variants only) ==========
F_price_only_1                 train MdAPE=4.8078%  holdout MdAPE=7.4378%
G_without_price_13             train MdAPE=4.6050%  holdout MdAPE=7.7235%
H_compact_no_price_4           train MdAPE=4.7557%  holdout MdAPE=7.5943%
I_price_and_dispersion_2       train MdAPE=4.7969%  holdout MdAPE=7.5862%

========== 2025-10 (new variants only) ==========
F_price_only_1                 train MdAPE=4.8122%  holdout MdAPE=7.2831%
G_without_price_13             train MdAPE=4.6266%  holdout MdAPE=7.3439%
H_compact_no_price_4           train MdAPE=4.7572%  holdout MdAPE=7.1119%
I_price_and_dispersion_2       train MdAPE=4.8091%  holdout MdAPE=7.3024%

========== 2025-11 (new variants only) ==========
F_price_only_1                 train MdAPE=4.8344%  holdout MdAPE=7.7803%
G_without_price_13             train MdAPE=4.6229%  holdout MdAPE=7.6590%
H_compact_no_price_4           train MdAPE=4.7537%  holdout MdAPE=7.7684%
I_price_and_dispersion_2       tr

In [8]:
ORIGINAL_PRED_PATH = OUTPUT / "predictions.csv"
if not ORIGINAL_PRED_PATH.exists():
    raise FileNotFoundError(
        f"Missing original predictions: {ORIGINAL_PRED_PATH}. "
        "Run the original ablation notebook once, or copy its predictions.csv here."
    )
original_predictions = pd.read_csv(ORIGINAL_PRED_PATH)
labels = ["A_full_14", "B_none_0", *FOLLOWUP_KEEP]
combined = pd.concat([
    original_predictions.loc[original_predictions.experiment.isin(labels[:2])],
    followup_predictions.loc[followup_predictions.experiment.isin(FOLLOWUP_KEEP)],
], ignore_index=True)

# A complete six-way comparison should have exactly one prediction per listing and variant.
assert not combined.duplicated(["month", "ml_num", "experiment"]).any(), "Duplicate predictions"
counts = combined.groupby("experiment").size()
expected_homes = sum(len(holdout) for _, holdout in folds.values())
if set(counts.index) != set(labels) or not (counts == expected_homes).all():
    raise ValueError(f"Incomplete/mismatched predictions. Expected {expected_homes} per variant; got {counts.to_dict()}")

# Validate the sold prices are identical for all variants for each home.
assert combined.groupby(["month", "ml_num"]).sold_price.nunique().max() == 1

summary_rows = []
month_rows = []
for label in labels:
    subset = combined[combined.experiment == label]
    summary_rows.append(dict(experiment=label, **followup_score(subset.sold_price, subset.prediction)))
    for month, rows in subset.groupby("month"):
        month_rows.append(dict(month=month, experiment=label,
                               **followup_score(rows.sold_price, rows.prediction)))
followup_pooled = pd.DataFrame(summary_rows).sort_values("mdape")
followup_monthly = pd.DataFrame(month_rows).sort_values(["month", "mdape"])
print("POOLED RESULTS (lower is better):")
display(followup_pooled)
print("MONTHLY RESULTS:")
display(followup_monthly)
followup_pooled.to_csv(FOLLOWUP_DIR / "pooled_comparison.csv", index=False)
followup_monthly.to_csv(FOLLOWUP_DIR / "monthly_comparison.csv", index=False)

paired = combined.pivot(index=["month", "ml_num", "sold_price"],
                        columns="experiment", values="prediction").reset_index()
paired = paired.dropna(subset=labels)
assert len(paired) == expected_homes, "Incomplete paired alignment"

def followup_bootstrap(candidate, baseline, sold_price, month, repeats=BOOTSTRAP_REPEATS):
    a = 100 * np.abs(candidate - sold_price) / sold_price
    b = 100 * np.abs(baseline - sold_price) / sold_price
    differences = np.empty((repeats, 2), dtype=float)
    rng = np.random.default_rng(SEED)
    groups = [np.flatnonzero(month == m) for m in np.unique(month)]
    for rep in range(repeats):
        sampled = np.concatenate([rng.choice(g, len(g), replace=True) for g in groups])
        differences[rep, 0] = a[sampled].mean() - b[sampled].mean()
        differences[rep, 1] = np.median(a[sampled]) - np.median(b[sampled])
    intervals = np.percentile(differences, [2.5, 97.5], axis=0)
    return {
        "delta_mape_pp": float(a.mean() - b.mean()),
        "mape_ci_low": float(intervals[0, 0]),
        "mape_ci_high": float(intervals[1, 0]),
        "delta_mdape_pp": float(np.median(a) - np.median(b)),
        "mdape_ci_low": float(intervals[0, 1]),
        "mdape_ci_high": float(intervals[1, 1]),
    }

comparisons = []
for experiment in FOLLOWUP_KEEP:
    for baseline in ("A_full_14", "B_none_0"):
        comparisons.append(dict(
            experiment=experiment, baseline=baseline, paired_homes=len(paired),
            **followup_bootstrap(
                paired[experiment].to_numpy(float), paired[baseline].to_numpy(float),
                paired.sold_price.to_numpy(float), paired.month.to_numpy(),
            ),
        ))
followup_comparisons = pd.DataFrame(comparisons)
print("PAIRED IMPROVEMENTS: negative means candidate is better")
display(followup_comparisons.sort_values(["baseline", "delta_mdape_pp"]))
followup_comparisons.to_csv(FOLLOWUP_DIR / "paired_bootstrap_vs_full_and_none.csv", index=False)

POOLED RESULTS (lower is better):


,experiment,homes,mape,mdape,p90_ape,median_signed_error
2,F_price_only_1,6152,9.878611,7.483836,21.202450,1.430746
1,B_none_0,6152,9.876657,7.493250,21.047365,1.680428
4,H_compact_no_price_4,6152,9.871941,7.540488,20.956802,1.505683
5,I_price_and_dispersion_2,6152,9.935138,7.559327,21.178293,1.484298
3,G_without_price_13,6152,10.055944,7.583098,21.456694,1.819404
0,A_full_14,6152,10.006191,7.645838,21.073595,1.479911


MONTHLY RESULTS:


,month,experiment,homes,mape,mdape,p90_ape,median_signed_error
6,2025-09,F_price_only_1,2035,9.854062,7.437809,21.223440,0.985442
15,2025-09,I_price_and_dispersion_2,2035,9.895934,7.586232,21.186474,0.954302
0,2025-09,A_full_14,2035,9.938776,7.586607,21.266341,0.571623
3,2025-09,B_none_0,2035,9.885724,7.590052,21.374868,1.420590
12,2025-09,H_compact_no_price_4,2035,9.867801,7.594303,20.875743,1.104360
9,2025-09,G_without_price_13,2035,10.072905,7.723549,21.510624,1.273790
13,2025-10,H_compact_no_price_4,2250,9.504237,7.111936,20.065341,0.744137
7,2025-10,F_price_only_1,2250,9.538741,7.283072,20.351793,0.283608
16,2025-10,I_price_and_dispersion_2,2250,9.597337,7.302356,20.378853,0.717255
4,2025-10,B_none_0,2250,9.578265,7.324047,19.750996,0.813851


PAIRED IMPROVEMENTS: negative means candidate is better


,experiment,baseline,paired_homes,delta_mape_pp,mape_ci_low,mape_ci_high,delta_mdape_pp,mdape_ci_low,mdape_ci_high
0,F_price_only_1,A_full_14,6152,-0.127580,-0.232756,-0.020476,-0.162002,-0.360195,0.033604
4,H_compact_no_price_4,A_full_14,6152,-0.134249,-0.246440,-0.019992,-0.105350,-0.315717,0.064026
6,I_price_and_dispersion_2,A_full_14,6152,-0.071052,-0.175880,0.029586,-0.086511,-0.276884,0.098898
2,G_without_price_13,A_full_14,6152,0.049754,-0.073109,0.163027,-0.062740,-0.258897,0.128154
1,F_price_only_1,B_none_0,6152,0.001954,-0.117545,0.123753,-0.009414,-0.235255,0.212623
5,H_compact_no_price_4,B_none_0,6152,-0.004716,-0.119267,0.111685,0.047238,-0.159323,0.238229
7,I_price_and_dispersion_2,B_none_0,6152,0.058481,-0.063472,0.183581,0.066077,-0.135058,0.285701
3,G_without_price_13,B_none_0,6152,0.179287,0.069876,0.286945,0.089848,-0.120643,0.283276


In [9]:
if followup_gain.empty:
    print("No new gain importance available (check previous failures).")
else:
    ranked = followup_gain.copy()
    ranked["gain_share"] = ranked.groupby(["month", "experiment"]).total_gain.transform(
        lambda s: s / s.sum() if s.sum() else s * 0
    )
    ranked = (ranked.groupby(["experiment", "feature"], as_index=False)
              .agg(mean_gain_share=("gain_share", "mean"),
                   mean_total_gain=("total_gain", "mean")))
    for label in FOLLOWUP_KEEP:
        print(f"\n{label} — top 15 features")
        display(ranked.loc[ranked.experiment == label]
                .sort_values("mean_gain_share", ascending=False).head(15))
    ranked.to_csv(FOLLOWUP_DIR / "gain_importance_F_to_I.csv", index=False)

print("All follow-up outputs:", FOLLOWUP_DIR)
print("Use Sep–Nov for selection only; do not use Dec–May to choose features.")


F_price_only_1 — top 15 features


,experiment,feature,mean_gain_share,mean_total_gain
2,F_price_only_1,comp_log_price,0.672220,130873.059896
1,F_price_only_1,community,0.114031,22200.162760
142,F_price_only_1,lot_front,0.059916,11665.222331
196,F_price_only_1,washrooms,0.039805,7748.103027
148,F_price_only_1,square_feet,0.017504,3409.524902
137,F_price_only_1,house_type_name,0.012219,2379.228353
140,F_price_only_1,lot_area,0.010881,2118.530802
141,F_price_only_1,lot_depth,0.006952,1353.105794
145,F_price_only_1,parking,0.003888,757.069173
147,F_price_only_1,season_sin,0.002546,496.049215



G_without_price_13 — top 15 features


,experiment,feature,mean_gain_share,mean_total_gain
351,G_without_price_13,lot_front,0.338392,66043.845052
357,G_without_price_13,square_feet,0.209263,40845.593750
198,G_without_price_13,community,0.176542,34455.912760
350,G_without_price_13,lot_depth,0.101810,19872.007161
405,G_without_price_13,washrooms,0.075205,14677.723307
349,G_without_price_13,lot_area,0.010316,2013.322713
346,G_without_price_13,house_type_name,0.009277,1811.117839
203,G_without_price_13,comp_log_ppsf,0.007715,1506.180664
354,G_without_price_13,parking,0.003792,740.238159
202,G_without_price_13,comp_local_ppsf_median,0.002887,563.577637



H_compact_no_price_4 — top 15 features


,experiment,feature,mean_gain_share,mean_total_gain
551,H_compact_no_price_4,lot_front,0.341298,66490.005208
557,H_compact_no_price_4,square_feet,0.211230,41154.576823
407,H_compact_no_price_4,community,0.180224,35110.997396
550,H_compact_no_price_4,lot_depth,0.087609,17065.798828
605,H_compact_no_price_4,washrooms,0.072267,14074.562174
409,H_compact_no_price_4,comp_log_ppsf,0.016985,3309.799154
546,H_compact_no_price_4,house_type_name,0.010505,2046.898560
549,H_compact_no_price_4,lot_area,0.010228,1992.600505
408,H_compact_no_price_4,comp_local_ppsf_median,0.003703,721.387207
554,H_compact_no_price_4,parking,0.003665,713.739115



I_price_and_dispersion_2 — top 15 features


,experiment,feature,mean_gain_share,mean_total_gain
608,I_price_and_dispersion_2,comp_log_price,0.656920,127899.252604
607,I_price_and_dispersion_2,community,0.114562,22303.990234
749,I_price_and_dispersion_2,lot_front,0.068567,13350.053385
803,I_price_and_dispersion_2,washrooms,0.032912,6407.649414
755,I_price_and_dispersion_2,square_feet,0.030020,5844.999186
744,I_price_and_dispersion_2,house_type_name,0.012464,2427.169596
747,I_price_and_dispersion_2,lot_area,0.010687,2080.981242
609,I_price_and_dispersion_2,comp_ppsf_std,0.005992,1166.967936
748,I_price_and_dispersion_2,lot_depth,0.004451,865.759786
752,I_price_and_dispersion_2,parking,0.003901,759.721456


All follow-up outputs: c:\Users\deant\Desktop\Coding\Data Dash final\outputs\experiments\comparable_feature_ablation_sep_nov_2025\price_feature_followup
Use Sep–Nov for selection only; do not use Dec–May to choose features.
